# WEEK 11 · 질문에 맞는 데이터 포스터 미션

이 노트북은 **범주별 프로그램 수 합계 가로 막대그래프**와 **시설 24개의 위치 좌표 산점도**를 한 장의 1600 × 2200 포스터로 만듭니다.

- 수정할 곳은 `STEP 1 · EDIT`와 `STEP 5 · EDIT` 두 셀뿐입니다.
- 나머지 셀은 위에서 아래로 실행하고 코드를 수정하지 않습니다.
- 마지막에 모든 초록 확인과 `WEEK 11 DATA POSTER COMPLETE`가 보이면 완료입니다.


In [ ]:
# STEP 0 · 실행 환경과 수업용 데이터 준비 — 이 셀은 수정하지 않습니다.
from pathlib import Path
import hashlib
import importlib.util
from importlib.metadata import PackageNotFoundError, version as package_version
import re
import subprocess
import sys

required_packages = {
    "pandas": ("pandas", "2.3.3"),
    "matplotlib": ("matplotlib", "3.10.8"),
    "seaborn": ("seaborn", "0.13.2"),
    "PIL": ("Pillow", "12.3.0"),
}
packages_to_install = []
for module_name, (package_name, required_version) in required_packages.items():
    try:
        installed_version = package_version(package_name)
    except PackageNotFoundError:
        installed_version = None
    if (
        importlib.util.find_spec(module_name) is None
        or installed_version != required_version
    ):
        packages_to_install.append(f"{package_name}=={required_version}")
if packages_to_install:
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-q", *packages_to_install]
    )

import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import font_manager
from matplotlib import image as mpimg
from matplotlib.colors import to_rgba
from matplotlib.markers import MarkerStyle
import seaborn as sns

try:
    from google.colab import files
except ImportError:
    files = None

mission_step0_execution = get_ipython().execution_count

SAMPLE_CSV_PATH = "week11_public_facilities_clean.csv"
SAMPLE_CSV = 'place_id,place_name,category,program_count,latitude,longitude\nC001,햇살도서관,도서관,48,37.5665,126.9780\nC002,나무도서관,도서관,35,37.5720,126.9900\nC003,구름도서관,도서관,62,37.5840,127.0120\nC004,샘물도서관,도서관,29,37.5510,126.9650\nC005,새봄도서관,도서관,54,37.5380,126.9920\nC006,한강도서관,도서관,41,37.5200,126.9400\nC007,별빛도서관,도서관,67,37.6030,127.0250\nC008,마루도서관,도서관,33,37.6120,126.9580\nC009,모양박물관,박물관,23,37.5790,126.9480\nC010,시간박물관,박물관,38,37.5900,126.9820\nC011,기록박물관,박물관,57,37.5610,127.0280\nC012,생활박물관,박물관,31,37.5430,127.0550\nC013,도시박물관,박물관,72,37.5280,127.0180\nC014,소리박물관,박물관,26,37.5110,126.9740\nC015,빛박물관,박물관,45,37.5960,127.0670\nC016,종이박물관,박물관,34,37.6170,127.0020\nC017,푸른문화센터,문화센터,52,37.5700,127.0440\nC018,열린문화센터,문화센터,64,37.5480,126.9250\nC019,다온문화센터,문화센터,28,37.5320,126.9550\nC020,누리문화센터,문화센터,49,37.5150,127.0410\nC021,이음문화센터,문화센터,70,37.5880,126.9300\nC022,마을문화센터,문화센터,37,37.6070,127.0480\nC023,함께문화센터,문화센터,58,37.6250,126.9850\nC024,오늘문화센터,문화센터,42,37.5020,127.0120\n'
EXPECTED_CSV_SHA256 = "dc0da6c249327470fa967dd0682eb0b0a62bd9f487ef014c2a8686db65d4cd94"
Path(SAMPLE_CSV_PATH).write_text(SAMPLE_CSV, encoding="utf-8")

dataset_title = "수업용 가상 공공문화시설 정제 데이터"
dataset_source = "Contents Programming Practice Week 11 · 교수자 제공 가상 자료"
dataset_license = "수업 목적 사용 허용"
reference_date = "2026-08-18"
observation_unit = "공공문화시설 한 곳"
expected_metadata = (
    dataset_title,
    dataset_source,
    dataset_license,
    reference_date,
    observation_unit,
)

POSTER_PAPER = "#f3efe5"
POSTER_INK = "#202523"
POSTER_MUTED = "#59615e"
POSTER_CORAL = "#a23d34"

category_markers = {
    "도서관": "o",
    "박물관": "s",
    "문화센터": "^",
}


def font_has_korean_glyphs(font_path):
    try:
        font = font_manager.get_font(font_path)
    except (OSError, RuntimeError):
        return False
    return all(
        font.get_char_index(ord(character))
        for character in "한글이름출처"
    )


def find_korean_font():
    preferred_tokens = (
        "nanumgothic",
        "notosanscjk",
        "notosanskr",
        "applesdgothic",
        "malgun",
        "pretendard",
    )
    supporting_fonts = [
        font_path
        for font_path in sorted(font_manager.findSystemFonts())
        if font_has_korean_glyphs(font_path)
    ]
    for font_path in supporting_fonts:
        compact_name = Path(font_path).name.lower().replace(" ", "")
        if any(token in compact_name for token in preferred_tokens):
            return font_path
    return supporting_fonts[0] if supporting_fonts else None


korean_font_path = find_korean_font()
if korean_font_path is None and Path("/etc/debian_version").exists():
    subprocess.check_call(["apt-get", "update", "-qq"])
    subprocess.check_call(["apt-get", "install", "-y", "-qq", "fonts-nanum"])
    korean_font_path = find_korean_font()

if korean_font_path is None:
    raise RuntimeError(
        "한글 글꼴을 찾지 못했습니다. Colab 새 런타임에서 STEP 0부터 다시 실행하세요."
    )

font_manager.fontManager.addfont(korean_font_path)
korean_font_name = font_manager.FontProperties(
    fname=korean_font_path
).get_name()
plt.rcParams["font.family"] = korean_font_name
plt.rcParams["axes.unicode_minus"] = False
sns.set_theme(style="whitegrid", font=korean_font_name)

print("준비 파일:", SAMPLE_CSV_PATH)
print("데이터:", dataset_title)
print("한글 글꼴:", korean_font_name)


## STEP 1 · 제출 정보와 시각 규칙

학번·이름을 입력하고, 데이터로 답할 수 있는 질문형 제목을 작성합니다. 세 범주에는 서로 다른 여섯 자리 HEX 색상을 지정합니다.


In [ ]:
# STEP 1 · EDIT — 학번·이름, 질문형 제목과 세 범주 색상을 수정합니다.
mission_step1_execution = get_ipython().execution_count

student_id = "학번"
student_name = "이름"
poster_title = "EDIT: 어느 시설 범주의 프로그램 수 합계가 큰가?"

category_palette = {
    "도서관": "#6b7280",
    "박물관": "#6b7280",
    "문화센터": "#6b7280",
}

print("제출자:", student_id, student_name)
print("포스터 질문:", poster_title)
print("범주 색상:", category_palette)


## STEP 2 · 정제 데이터 확인

수업용 24행 CSV를 불러오고 열, 행, 범주, 좌표의 유효성을 확인합니다. 제공 파일과 DataFrame은 수정하지 않습니다.


In [ ]:
# STEP 2 · 정제 CSV 불러오기와 구조 확인 — 이 셀은 수정하지 않습니다.
mission_step2_execution = get_ipython().execution_count

source_path = Path(SAMPLE_CSV_PATH)
source_bytes_before = source_path.read_bytes()
assert (
    hashlib.sha256(source_bytes_before).hexdigest() == EXPECTED_CSV_SHA256
), "제공 CSV 내용이 수업 기준과 다릅니다. 새 런타임에서 다시 실행하세요."
facility_df = pd.read_csv(source_path)
source_snapshot = facility_df.copy(deep=True)

required_columns = {
    "place_id",
    "place_name",
    "category",
    "program_count",
    "latitude",
    "longitude",
}
missing_columns = sorted(required_columns - set(facility_df.columns))
if missing_columns:
    raise KeyError("필요한 열이 없습니다: " + ", ".join(missing_columns))

assert len(facility_df) == 24, "수업용 정제 데이터는 24행이어야 합니다."
assert facility_df["category"].nunique() == 3, "시설 범주는 세 개여야 합니다."
assert facility_df["place_id"].nunique() == 24, "place_id가 중복되었습니다."
assert facility_df[["program_count", "latitude", "longitude"]].notna().all().all()
assert facility_df["latitude"].between(-90, 90).all()
assert facility_df["longitude"].between(-180, 180).all()

print("데이터 크기:", facility_df.shape)
print("범주별 시설 수:")
print(facility_df["category"].value_counts().sort_index())
print("앞 5행:")
print(facility_df.head().to_string(index=False))


## STEP 3 · 범주별 합계

시설 24행을 세 범주로 묶고 프로그램 수를 더한 뒤 작은 값부터 정렬합니다.


In [ ]:
# STEP 3 · 범주별 프로그램 수 합계 만들기 — 이 셀은 수정하지 않습니다.
mission_step3_execution = get_ipython().execution_count

category_summary = (
    facility_df
    .groupby("category", as_index=False)["program_count"]
    .sum()
    .sort_values("program_count")
    .reset_index(drop=True)
)

expected_totals = {
    "박물관": 326,
    "도서관": 369,
    "문화센터": 400,
}
actual_totals = dict(
    zip(
        category_summary["category"],
        category_summary["program_count"],
    )
)
assert actual_totals == expected_totals, "범주별 합계가 수업 기준과 다릅니다."

print(category_summary.to_string(index=False))


## STEP 4 · 두 그래프

위쪽 Axes에는 0에서 시작하는 막대 세 개를, 아래쪽 Axes에는 시설 24개의 좌표 점을 만듭니다. 좌표 점의 색상과 표식은 범주, 면적은 프로그램 수를 나타냅니다.


In [ ]:
# STEP 4 · 가로 막대그래프와 위치 좌표 산점도 만들기 — 이 셀은 수정하지 않습니다.
mission_step4_execution = get_ipython().execution_count

fig, axes = plt.subplots(
    nrows=2,
    ncols=1,
    figsize=(8, 11),
    gridspec_kw={"height_ratios": [0.82, 1.28]},
)
fig.patch.set_facecolor(POSTER_PAPER)
fig.subplots_adjust(
    left=0.14,
    right=0.74,
    top=0.77,
    bottom=0.18,
    hspace=0.58,
)

sns.barplot(
    data=category_summary,
    x="program_count",
    y="category",
    hue="category",
    palette=category_palette,
    errorbar=None,
    legend=False,
    edgecolor=POSTER_INK,
    ax=axes[0],
)
axes[0].set_xlim(left=0)
axes[0].set_xlabel("프로그램 수 합계")
axes[0].set_ylabel("")
axes[0].set_title(
    "어느 시설 범주의 프로그램 수 합계가 큰가?",
    loc="left",
    fontweight="bold",
)

for bar in axes[0].patches:
    value = int(round(bar.get_width()))
    axes[0].text(
        value + 6,
        bar.get_y() + bar.get_height() / 2,
        str(value),
        va="center",
        fontweight="bold",
    )

bar_count = len(axes[0].patches)
bar_axis_left_limit = axes[0].get_xlim()[0]

scatter_plot = sns.scatterplot(
    data=facility_df,
    x="longitude",
    y="latitude",
    hue="category",
    style="category",
    size="program_count",
    palette=category_palette,
    markers=category_markers,
    sizes=(60, 300),
    alpha=0.82,
    edgecolor=POSTER_INK,
    linewidth=0.8,
    legend="brief",
    ax=axes[1],
)
axes[1].set_xlabel("경도")
axes[1].set_ylabel("위도")
axes[1].set_title(
    "시설 24개는 서로 어디에 놓였는가?",
    loc="left",
    fontweight="bold",
)
axes[1].set_xticks([126.90, 126.95, 127.00, 127.05, 127.10])
axes[1].set_aspect("equal", adjustable="datalim")
scatter_legend = axes[1].legend(
    bbox_to_anchor=(1.02, 1.0),
    loc="upper left",
    borderaxespad=0,
    frameon=True,
    framealpha=0.94,
    fontsize=8,
)
legend_label_map = {
    "category": "시설 범주",
    "program_count": "프로그램 수",
}
for legend_text in scatter_legend.get_texts():
    legend_text.set_text(
        legend_label_map.get(legend_text.get_text(), legend_text.get_text())
    )

plotted_collections = [
    collection
    for collection in axes[1].collections
    if len(collection.get_offsets()) > 0
]
plotted_offsets = [
    collection.get_offsets() for collection in plotted_collections
]
scatter_point_count = sum(len(offsets) for offsets in plotted_offsets)


def marker_path_signature(path):
    return (
        path.codes.tobytes() if path.codes is not None else b"",
        path.vertices.round(6).tobytes(),
    )


scatter_unique_sizes = len({
    round(float(size), 6)
    for collection in plotted_collections
    for size in collection.get_sizes()
})
scatter_unique_colors = len({
    tuple(round(float(channel), 6) for channel in color)
    for collection in plotted_collections
    for color in collection.get_facecolors()
})
scatter_marker_signatures = {
    marker_path_signature(path)
    for collection in plotted_collections
    for path in collection.get_paths()
}
scatter_unique_markers = len(scatter_marker_signatures)
primary_scatter_collection = (
    plotted_collections[0] if len(plotted_collections) == 1 else None
)
if primary_scatter_collection is None:
    actual_scatter_offsets = []
    actual_scatter_colors = []
    actual_scatter_markers = []
    actual_scatter_sizes = []
else:
    actual_scatter_offsets = [
        tuple(round(float(coordinate), 6) for coordinate in point)
        for point in primary_scatter_collection.get_offsets()
    ]
    actual_scatter_colors = [
        tuple(round(float(channel), 6) for channel in color)
        for color in primary_scatter_collection.get_facecolors()
    ]
    actual_scatter_markers = [
        marker_path_signature(path)
        for path in primary_scatter_collection.get_paths()
    ]
    actual_scatter_sizes = [
        round(float(size), 6)
        for size in primary_scatter_collection.get_sizes()
    ]

expected_scatter_offsets = [
    (round(float(longitude), 6), round(float(latitude), 6))
    for longitude, latitude in zip(
        facility_df["longitude"],
        facility_df["latitude"],
    )
]
expected_scatter_colors = [
    tuple(
        round(float(channel), 6)
        for channel in to_rgba(category_palette[category], alpha=0.82)
    )
    for category in facility_df["category"]
]
expected_marker_signatures = {}
for category, marker_symbol in category_markers.items():
    marker_style = MarkerStyle(marker_symbol)
    marker_path = marker_style.get_path().transformed(
        marker_style.get_transform()
    )
    expected_marker_signatures[category] = marker_path_signature(marker_path)
expected_scatter_markers = [
    expected_marker_signatures[category]
    for category in facility_df["category"]
]
program_counts = facility_df["program_count"].tolist()
expected_size_order = sorted(
    range(len(program_counts)),
    key=program_counts.__getitem__,
)
actual_size_order = sorted(
    range(len(actual_scatter_sizes)),
    key=actual_scatter_sizes.__getitem__,
)
scatter_offsets_match_rows = actual_scatter_offsets == expected_scatter_offsets
scatter_colors_follow_category = actual_scatter_colors == expected_scatter_colors
scatter_markers_follow_category = actual_scatter_markers == expected_scatter_markers
scatter_sizes_follow_program_count = (
    len(actual_scatter_sizes) == len(program_counts)
    and actual_size_order == expected_size_order
)
plotted_place_ids = facility_df["place_id"].tolist()

print("막대 수:", bar_count)
print("좌표 점 수:", scatter_point_count)
print(
    "좌표 표현:",
    f"색상 {scatter_unique_colors} · 표식 {scatter_unique_markers} · "
    f"크기 단계 {scatter_unique_sizes}",
)


## STEP 5 · 관찰과 한계

그래프에서 실제로 확인한 합계를 포함해 관찰 문장을 작성합니다. 그다음 가상 자료와 좌표 그래프로 단정할 수 없는 내용을 한계로 적습니다.


In [ ]:
# STEP 5 · EDIT — 그래프에서 확인한 관찰과 해석의 한계를 작성합니다.
mission_step5_execution = get_ipython().execution_count

main_observation = (
    "EDIT: 합계 326·369·400 중 하나를 근거로 30자 이상 관찰하세요."
)
limitation_statement = (
    "EDIT: 가상 자료만으로 단정할 수 없는 내용을 30자 이상 적으세요."
)

print("관찰:", main_observation)
print("한계:", limitation_statement)


## STEP 6 · 포스터 저장

제목, 두 그래프, 관찰, 한계, 출처를 Figure 안에 배치하고 1600 × 2200 PNG로 저장합니다.


In [ ]:
# STEP 6 · 포스터 설명 배치와 1600 × 2200 PNG 저장 — 이 셀은 수정하지 않습니다.
mission_step6_execution = get_ipython().execution_count

safe_student_id = str(student_id).strip()
safe_student_name = str(student_name).strip()
if "\n" in poster_title or "\r" in poster_title:
    raise AssertionError("질문형 제목은 줄바꿈 없이 한 줄로 작성하세요.")
safe_name_pattern = re.compile(r"^[0-9A-Za-z가-힣_-]+$")
if not (
    safe_name_pattern.fullmatch(safe_student_id)
    and safe_name_pattern.fullmatch(safe_student_name)
):
    raise AssertionError(
        "학번과 이름에는 한글·영문·숫자·밑줄·하이픈만 사용할 수 있습니다."
    )
input_text_lengths_safe = (
    len(poster_title.strip()) <= 50
    and len(main_observation.strip()) <= 90
    and len(limitation_statement.strip()) <= 90
)
if not input_text_lengths_safe:
    raise AssertionError(
        "제목은 50자, 관찰과 한계는 각각 90자 이내로 다듬어 주세요."
    )
output_filename = (
    f"week11_{safe_student_id}_{safe_student_name}_data_poster.png"
)

title_artist = fig.suptitle(
    poster_title,
    x=0.10,
    y=0.95,
    ha="left",
    fontsize=22,
    fontweight="bold",
)
subtitle_artist = fig.text(
    0.10,
    0.865,
    "같은 24행 데이터를 범주별 합계와 상대적 위치로 다시 읽기",
    fontsize=11,
    color=POSTER_MUTED,
)
observation_label_artist = fig.text(
    0.10, 0.125, "핵심 관찰", fontsize=10, fontweight="bold", color=POSTER_CORAL
)
observation_artist = fig.text(
    0.10, 0.098, main_observation, fontsize=9.5, wrap=True
)
limitation_label_artist = fig.text(
    0.10, 0.068, "해석의 한계", fontsize=10, fontweight="bold", color=POSTER_CORAL
)
limitation_artist = fig.text(
    0.10, 0.041, limitation_statement, fontsize=9.5, wrap=True
)
source_artist = fig.text(
    0.10,
    0.012,
    f"출처 · {dataset_source} · 기준일 {reference_date} · {len(facility_df)}행",
    fontsize=7.5,
    color=POSTER_MUTED,
)

fig.canvas.draw()
renderer = fig.canvas.get_renderer()
figure_bounds = fig.bbox
poster_text_artists = (
    title_artist,
    subtitle_artist,
    observation_label_artist,
    observation_artist,
    limitation_label_artist,
    limitation_artist,
    source_artist,
)
poster_text_inside_canvas = all(
    text_artist.get_window_extent(renderer).x0 >= figure_bounds.x0
    and text_artist.get_window_extent(renderer).y0 >= figure_bounds.y0
    and text_artist.get_window_extent(renderer).x1 <= figure_bounds.x1
    and text_artist.get_window_extent(renderer).y1 <= figure_bounds.y1
    for text_artist in poster_text_artists
)
observation_bounds = observation_artist.get_window_extent(renderer)
limitation_label_bounds = limitation_label_artist.get_window_extent(renderer)
limitation_bounds = limitation_artist.get_window_extent(renderer)
source_bounds = source_artist.get_window_extent(renderer)
footer_blocks_separated = (
    observation_bounds.y0 > limitation_label_bounds.y1
    and limitation_bounds.y0 > source_bounds.y1
)
if not poster_text_inside_canvas or not footer_blocks_separated:
    raise AssertionError(
        "제목·관찰·한계가 포스터 경계를 넘거나 서로 겹칩니다. 문장을 줄여 주세요."
    )

fig.savefig(
    output_filename,
    dpi=200,
    facecolor=POSTER_PAPER,
)
output_path = Path(output_filename)
output_bytes = output_path.read_bytes()
saved_image = mpimg.imread(output_path)

print("저장 파일:", output_filename)
print("저장 크기:", saved_image.shape[:2])
plt.show()


## STEP 7 · 자동 검사와 내려받기

수정하지 않습니다. 실패한 조건의 한글 설명을 읽고 STEP 1 또는 STEP 5만 고친 뒤, 새 런타임에서 모두 실행합니다.


In [ ]:
# STEP 7 · FINAL CHECK — 이 셀은 수정하지 않습니다.
mission_step7_execution = get_ipython().execution_count

execution_sequence = (
    mission_step0_execution,
    mission_step1_execution,
    mission_step2_execution,
    mission_step3_execution,
    mission_step4_execution,
    mission_step5_execution,
    mission_step6_execution,
    mission_step7_execution,
)
hex_color_pattern = re.compile(r"^#[0-9A-Fa-f]{6}$")
palette_values = list(category_palette.values())
title_data_terms = (
    "프로그램",
    "범주",
    "합계",
    "개수",
    "시설 수",
    "위도",
    "경도",
    "좌표",
    "위치",
    "어디",
)
common_unsupported_title_terms = (
    "좋은",
    "최고",
    "인기",
    "만족",
    "추천",
    "유익",
    "우수",
    "효율",
)
title_has_data_clue = (
    any(term in poster_title for term in title_data_terms)
    and not any(term in poster_title for term in common_unsupported_title_terms)
)
current_metadata = (
    dataset_title,
    dataset_source,
    dataset_license,
    reference_date,
    observation_unit,
)

checks = [
    (
        execution_sequence == (1, 2, 3, 4, 5, 6, 7, 8),
        "새 런타임에서 STEP 0부터 여덟 셀을 순서대로 실행",
    ),
    (
        safe_student_id != ""
        and safe_student_name != ""
        and "학번" not in safe_student_id
        and "이름" not in safe_student_name
        and safe_name_pattern.fullmatch(safe_student_id)
        and safe_name_pattern.fullmatch(safe_student_name),
        "학번·이름과 안전한 파일명",
    ),
    (
        not poster_title.strip().startswith("EDIT:")
        and len(poster_title.strip()) >= 15
        and len(poster_title.strip()) <= 50
        and poster_title.strip().endswith(("?", "？"))
        and title_has_data_clue,
        "15–50자이며 데이터 단서를 포함한 질문형 제목 형식",
    ),
    (
        len(palette_values) == 3
        and len(set(palette_values)) == 3
        and all(hex_color_pattern.fullmatch(color) for color in palette_values),
        "세 범주의 서로 다른 HEX 색상",
    ),
    (
        source_path.read_bytes() == source_bytes_before
        and hashlib.sha256(source_bytes_before).hexdigest()
        == EXPECTED_CSV_SHA256
        and facility_df.equals(source_snapshot),
        "제공 CSV와 불러온 24행 원본 보존",
    ),
    (
        len(facility_df) == 24
        and facility_df["category"].nunique() == 3
        and facility_df["place_id"].nunique() == 24,
        "시설 24행과 세 범주",
    ),
    (
        actual_totals == {"박물관": 326, "도서관": 369, "문화센터": 400},
        "범주별 프로그램 수 합계 326·369·400",
    ),
    (
        bar_count == 3 and abs(bar_axis_left_limit) < 1e-9,
        "0에서 시작하는 막대 세 개",
    ),
    (
        scatter_point_count == len(facility_df) == 24
        and len(plotted_place_ids) == len(set(plotted_place_ids)) == 24,
        "정제 24행과 좌표 점 24개",
    ),
    (
        scatter_unique_colors == 3
        and scatter_unique_markers == 3
        and scatter_unique_sizes > 1,
        "색상·표식·크기로 구분한 좌표 점",
    ),
    (
        scatter_offsets_match_rows
        and scatter_colors_follow_category
        and scatter_markers_follow_category
        and scatter_sizes_follow_program_count,
        "원본 데이터와 색상·표식·크기의 대응",
    ),
    (
        not main_observation.strip().startswith("EDIT:")
        and len(main_observation.strip()) >= 30
        and re.search(r"326|369|400", main_observation),
        "실제 합계를 포함한 30자 이상의 관찰 문장",
    ),
    (
        not limitation_statement.strip().startswith("EDIT:")
        and len(limitation_statement.strip()) >= 30,
        "30자 이상의 해석 한계",
    ),
    (
        input_text_lengths_safe
        and poster_text_inside_canvas
        and footer_blocks_separated,
        "글자 수와 포스터 경계 안의 제목·관찰·한계",
    ),
    (
        output_path.exists()
        and len(output_bytes) > 50000
        and saved_image.shape[:2] == (2200, 1600),
        "1600 × 2200 데이터 포스터 PNG",
    ),
    (
        current_metadata == expected_metadata,
        "출처·이용 조건·기준일·관찰 단위",
    ),
]

failed_checks = []
for passed, label in checks:
    if passed:
        print("✅", label)
    else:
        print("❌", label)
        failed_checks.append(label)

if failed_checks:
    raise AssertionError(
        "위의 빨간 조건을 수정한 뒤 새 런타임에서 모두 실행하세요: "
        + ", ".join(failed_checks)
    )

print("🎉 WEEK 11 DATA POSTER COMPLETE")
if files is not None:
    files.download(output_filename)
